# Preparazione CIC-IDS2017 per il confronto storico Autoencoder_2.0

Versione autonoma e snellita. Conserva le stesse regole: rimozione di righe-header duplicate, NaN/Inf e negativi non validi; gestione delle sentinelle IAT -1; selezione ordinata di 59 feature piu Label. Per il 2018 conserva la rinomina allo schema 2017. Questo schema CIC di confronto e distinto dal PFS29.

La modalita predefinita report legge soltanto gli header finali. test usa fixture sintetiche; prepare ricostruisce i CSV clean e final. Gli originali restano intatti; il rebuild sostituisce i file derivati. Non eseguire prepare per validare questa riorganizzazione.


In [ ]:
from pathlib import Path
import os
import json
import hashlib
import numpy as np
import pandas as pd
from IPython.display import HTML, display

# Point to the CSV parent folder when notebooks and datasets are stored separately.
NOTEBOOK_DIR = Path('.').resolve()
ROOT = Path(os.environ.get('NIDS_PROJECT_ROOT', str(NOTEBOOK_DIR))).resolve()
NOTEBOOK_NAMES = ('Autoencoder_2.0.ipynb', 'Autoencoder_NetFlow_Multidomain.ipynb',
                  'Dataset.ipynb', 'Dataset_2018.ipynb', 'Dataset_NetFlow_Preparation.ipynb')
def show_table(rows, title):
    print(title)
    display(pd.DataFrame(rows))
CIC_PREPARATION_MODE = 'report'  # 'report', 'test', 'prepare'
RAW_FOLDER = 'dataset'
FINAL_FOLDER = 'dataset_final_2017'


## Schema congelato e pulizia dei valori negativi

In [ ]:
# ============================================================
# SCHEMA FINALE COMUNE 2017/2018
# 59 feature + Label = 60 colonne
# ============================================================

FINAL_COLUMNS = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",

    "Fwd Packet Length Max",
    "Fwd Packet Length Min",
    "Fwd Packet Length Mean",
    "Fwd Packet Length Std",

    "Bwd Packet Length Max",
    "Bwd Packet Length Min",
    "Bwd Packet Length Mean",
    "Bwd Packet Length Std",

    "Flow Bytes/s",
    "Flow Packets/s",

    "Flow IAT Mean",
    "Flow IAT Std",
    "Flow IAT Max",
    "Flow IAT Min",

    "Fwd IAT Total",
    "Fwd IAT Mean",
    "Fwd IAT Std",
    "Fwd IAT Max",
    "Fwd IAT Min",

    "Bwd IAT Total",
    "Bwd IAT Mean",
    "Bwd IAT Std",
    "Bwd IAT Max",
    "Bwd IAT Min",

    "Fwd Header Length",
    "Bwd Header Length",

    "Fwd Packets/s",
    "Bwd Packets/s",

    "Min Packet Length",
    "Max Packet Length",
    "Packet Length Mean",
    "Packet Length Std",
    "Packet Length Variance",

    "FIN Flag Count",
    "SYN Flag Count",
    "RST Flag Count",
    "PSH Flag Count",
    "ACK Flag Count",
    "URG Flag Count",
    "ECE Flag Count",

    "Down/Up Ratio",
    "Average Packet Size",

    "Init_Win_bytes_forward",
    "Init_Win_bytes_backward",
    "act_data_pkt_fwd",
    "min_seg_size_forward",

    "Active Mean",
    "Active Std",
    "Active Max",
    "Active Min",

    "Idle Mean",
    "Idle Std",
    "Idle Max",
    "Idle Min",

    "Label",
]
# ============================================================
# PULIZIA VALORI NEGATIVI FISICAMENTE NON VALIDI
# Regola simmetrica 2017/2018, prima della selezione FINAL_COLUMNS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

NEGATIVE_AUDIT_DIR = ROOT / "Artefatti" / "preprocessing_negative_cleaning"


NEGATIVE_PHYSICAL_COLUMNS = [
    "Flow Duration",
    "Flow IAT Mean",
    "Flow IAT Std",
    "Flow IAT Max",
    "Flow IAT Min",
    "Fwd IAT Total",
    "Fwd IAT Mean",
    "Fwd IAT Std",
    "Fwd IAT Max",
    "Fwd IAT Min",
    "Bwd IAT Total",
    "Bwd IAT Mean",
    "Bwd IAT Std",
    "Bwd IAT Max",
    "Bwd IAT Min",
    "Flow Bytes/s",
    "Flow Packets/s",
]

NEGATIVE_RATE_OR_DURATION_COLUMNS = [
    "Flow Duration",
    "Flow Bytes/s",
    "Flow Packets/s",
]

IAT_SENTINEL_MINUS_ONE_COLUMNS = [
    "Flow IAT Min",
    "Fwd IAT Min",
    "Bwd IAT Min",
]


def _numeric_series(df: pd.DataFrame, column: str) -> pd.Series:
    return pd.to_numeric(df[column], errors="coerce")


def clean_negative_timing_values(df: pd.DataFrame, file_name: str, dataset_version: str) -> pd.DataFrame:
    """Gestisce negativi finiti prima di FINAL_COLUMNS senza usare label per decidere.

    Regola congelata dopo Fase 0:
    - drop righe con durata/rate negativi o IAT negativi diversi da -1;
    - sostituisce a 0 i -1 residui nelle colonne IAT min, interpretati come sentinella.
    """
    rows_before = len(df)
    available_negative_cols = [c for c in NEGATIVE_PHYSICAL_COLUMNS if c in df.columns]

    any_negative_mask = pd.Series(False, index=df.index)
    corrupt_mask = pd.Series(False, index=df.index)
    sentinel_cell_count = 0
    sentinel_row_mask = pd.Series(False, index=df.index)
    negative_cell_count = 0

    for col in available_negative_cols:
        values = _numeric_series(df, col)
        neg_mask = values < 0
        neg_count = int(neg_mask.sum())
        negative_cell_count += neg_count
        any_negative_mask |= neg_mask

        if col in NEGATIVE_RATE_OR_DURATION_COLUMNS:
            corrupt_mask |= neg_mask
        elif col in IAT_SENTINEL_MINUS_ONE_COLUMNS:
            sentinel_mask = values.eq(-1)
            non_sentinel_negative_mask = neg_mask & ~sentinel_mask
            corrupt_mask |= non_sentinel_negative_mask
            sentinel_cell_count += int(sentinel_mask.sum())
            sentinel_row_mask |= sentinel_mask
        else:
            corrupt_mask |= neg_mask

    dropped_df = df.loc[corrupt_mask]
    df_clean = df.loc[~corrupt_mask].copy()

    corrected_cells = 0
    corrected_rows_mask = pd.Series(False, index=df_clean.index)
    for col in IAT_SENTINEL_MINUS_ONE_COLUMNS:
        if col not in df_clean.columns:
            continue
        values = _numeric_series(df_clean, col)
        fix_mask = values.eq(-1)
        corrected_cells += int(fix_mask.sum())
        corrected_rows_mask |= fix_mask
        if fix_mask.any():
            df_clean.loc[fix_mask, col] = 0

    rows_after = len(df_clean)
    touched_rows = int(any_negative_mask.sum())
    dropped_rows = int(corrupt_mask.sum())
    corrected_rows = int(corrected_rows_mask.sum())

    dropped_labels = ""
    corrected_labels = ""
    if "Label" in df.columns and dropped_rows:
        dropped_labels = "; ".join(
            f"{label}:{count}"
            for label, count in dropped_df["Label"].astype(str).value_counts().items()
        )
    if "Label" in df_clean.columns and corrected_rows:
        corrected_labels = "; ".join(
            f"{label}:{count}"
            for label, count in df_clean.loc[corrected_rows_mask, "Label"].astype(str).value_counts().items()
        )

    report_row = {
        "dataset_version": dataset_version,
        "file": file_name,
        "rows_before": rows_before,
        "rows_after": rows_after,
        "rows_touched_by_any_negative": touched_rows,
        "negative_cells_total": negative_cell_count,
        "sentinel_minus_one_cells_seen": sentinel_cell_count,
        "rows_dropped_corrupt": dropped_rows,
        "rows_corrected_minus_one_to_zero": corrected_rows,
        "cells_corrected_minus_one_to_zero": corrected_cells,
        "dropped_label_distribution_diagnostic": dropped_labels,
        "corrected_label_distribution_diagnostic": corrected_labels,
    }

    report_path = NEGATIVE_AUDIT_DIR / f"{dataset_version}_negative_cleaning_report.csv"
    pd.DataFrame([report_row]).to_csv(
        report_path,
        mode="a",
        index=False,
        header=not report_path.exists(),
    )

    print(
        "  [NEGATIVE CLEAN] "
        f"touched_rows={touched_rows:,} "
        f"dropped={dropped_rows:,} "
        f"corrected_rows={corrected_rows:,} "
        f"corrected_cells={corrected_cells:,} "
        f"{rows_before:,}->{rows_after:,}"
    )
    if dropped_labels:
        print(f"    dropped label audit: {dropped_labels}")
    if corrected_labels:
        print(f"    corrected label audit: {corrected_labels}")

    return df_clean


## Verifica sintetica e report senza rebuild

In [ ]:
import tempfile
from unittest.mock import patch

if CIC_PREPARATION_MODE not in {'report', 'test', 'prepare'}:
    raise ValueError('Usare report, test o prepare.')
assert len(FINAL_COLUMNS) == 60 and len(set(FINAL_COLUMNS)) == 60
assert FINAL_COLUMNS[-1] == 'Label'
if CIC_PREPARATION_MODE == 'test':
    fixture = pd.DataFrame({'Flow Duration':[10, -1, 10, 10, 10],
        'Flow Bytes/s':[1, 1, -1, 1, 1], 'Flow IAT Min':[0, 0, 0, -1, -2],
        'Label':['BENIGN','A','B','C','D']})
    with tempfile.TemporaryDirectory() as temporary:
        with patch.dict(globals(), NEGATIVE_AUDIT_DIR=Path(temporary)):
            cleaned = clean_negative_timing_values(fixture, 'synthetic.csv', 'synthetic')
            assert cleaned['Label'].tolist() == ['BENIGN','C']
            assert cleaned.loc[3,'Flow IAT Min'] == 0
    assert fixture.loc[3,'Flow IAT Min'] == -1
    print('STANDALONE CIC TEST COMPLETATO: schema e regole congelate; nessun CSV reale modificato.')
elif CIC_PREPARATION_MODE == 'report':
    rows = []
    for file in sorted((ROOT/FINAL_FOLDER).glob('*.csv')):
        columns = pd.read_csv(file, nrows=0).columns.tolist()
        rows.append({'file':file.name, 'columns':len(columns), 'schema_matches':columns == FINAL_COLUMNS})
    show_table(rows, 'Header dei CSV di confronto gia preparati')
    if not rows:
        print('CSV finali assenti: report disponibile quando vengono forniti i dataset.')
    elif not all(r['schema_matches'] for r in rows):
        raise ValueError('Schema finale diverso dal confronto storico.')

## Ricostruzione esplicita dei CSV di confronto

In [ ]:
if CIC_PREPARATION_MODE == 'prepare':
    if not sorted((ROOT / RAW_FOLDER).glob('*.csv')):
        raise FileNotFoundError('CSV originali mancanti: '+str(ROOT / RAW_FOLDER))
    NEGATIVE_AUDIT_DIR.mkdir(parents=True, exist_ok=True)
    # ============================================================
    # PULIZIA DATASET 2017 DA NaN / INF
    # Input : dataset
    # Output: dataset_clean
    # ============================================================
    
    dataset_old_dir = (ROOT / "dataset")
    dataset_dir = (ROOT / "dataset_clean")
    dataset_dir.mkdir(exist_ok=True)
    
    for file in dataset_old_dir.glob("*.csv"):
        print(f"Processing {file.name}")
    
        df = pd.read_csv(file, low_memory=False)
        df.columns = df.columns.str.strip()
    
        if "Label" in df.columns:
            df = df[df["Label"] != "Label"]
    
        rows_before = len(df)
    
        df_clean = (
            df.replace([np.inf, -np.inf], np.nan)
              .dropna()
        )
    
        rows_after = len(df_clean)
    
        df_clean.to_csv(dataset_dir / file.name, index=False)
    
        print(
            f"  [SAVED] {file.name} "
            f"{rows_before:,} -> {rows_after:,} "
            f"shape={df_clean.shape}"
        )
    # ============================================================
    # ALLINEAMENTO FINALE DATASET 2017
    # Input : dataset_clean
    # Output: dataset_final_2017
    #
    # Qui NON facciamo solo drop di alcune colonne.
    # Qui selezioniamo rigidamente le 60 colonne finali:
    # 59 feature + Label.
    # ============================================================
    
    dataset_dir_2017 = (ROOT / "dataset_clean")
    output_dir_2017 = (ROOT / "dataset_final_2017")
    output_dir_2017.mkdir(exist_ok=True)
    
    negative_report_2017 = NEGATIVE_AUDIT_DIR / "2017_negative_cleaning_report.csv"
    if negative_report_2017.exists():
        negative_report_2017.unlink()
    
    for csv_file in dataset_dir_2017.glob("*.csv"):
        print(f"Processing {csv_file.name}")
    
        df = pd.read_csv(csv_file, low_memory=False)
        df.columns = df.columns.str.strip()
    
        # Difesa contro righe-header duplicate
        if "Label" in df.columns:
            df = df[df["Label"] != "Label"]
    
        # Difesa contro eventuali colonne duplicate nel CSV
        df = df.loc[:, ~df.columns.duplicated()]
    
        df = clean_negative_timing_values(df, csv_file.name, "2017")
    
        missing = [c for c in FINAL_COLUMNS if c not in df.columns]
        extra = [c for c in df.columns if c not in FINAL_COLUMNS]
    
        if missing:
            raise ValueError(f"{csv_file.name} - colonne mancanti: {missing}")
    
        if extra:
            print(f"  Scartate colonne extra: {extra}")
    
        # Riga fondamentale:
        # seleziona e riordina esattamente le colonne finali
        df = df[FINAL_COLUMNS]
    
        output_path = output_dir_2017 / csv_file.name
        df.to_csv(output_path, index=False)
    
        print(f"  Colonne finali: {len(df.columns)}  shape={df.shape}")
